# Yellow Taxi 2009-01 — Ingestion-Vergleich: Spark vs. DuckDB

Zwei Ansaetze fuer denselben Raw-Layer-Ingestion-Job, jeweils mit Zeitmessung:

1. **Spark** -> `nessie.raw.yellow_taxi_200901_spark` (produktionsreifer Write-Pfad ueber Iceberg-Spark-Runtime + Nessie-Katalog)
2. **DuckDB** -> `nessie.raw.yellow_taxi_200901_duck` (experimenteller Write-Pfad via DuckDB `iceberg`-Extension + Nessie REST Catalog)

> Hinweis: DuckDB's Iceberg-Schreibpfad ist laut DuckDB-Doku aktuell experimentell.
> Dieser Vergleich dient der Einordnung (Geschwindigkeit, Handhabung), nicht als
> Empfehlung, DuckDB in produktiven ETL-Jobs als Spark-Ersatz einzusetzen.

In [24]:
import os
import time

DATA_DIR = "../data/yellow_taxi"
SOURCE_FILE = f"{DATA_DIR}/yellow_tripdata_2009-01.parquet"

RESULTS = {}  # {"approach": {"table": ..., "rows": ..., "seconds": ...}}

def log(msg: str) -> None:
    print(f"  {msg}", flush=True)

print("Ready")

Ready


## Approach 1: Spark -> Iceberg via Nessie

Nutzt die im Cluster bereits konfigurierte `iceberg-spark-runtime` + `nessie`-Katalog-
Konfiguration (analog zu `spark-ingestion.py`). Es wird angenommen, dass
`spark-defaults.conf` bereits `spark.sql.catalog.nessie` etc. setzt.

In [25]:
from pyspark.sql import SparkSession

SPARK_TABLE = "nessie.raw.yellow_taxi_200901_spark"
SPARK_RAW_BUCKET = "s3a://raw"

print("\n[SPARK] SparkSession starten ...", flush=True)
spark = (
    SparkSession.builder
    .appName("yellow-taxi-200901-spark-ingestion")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("\n[SPARK] Namespace und Cleanup ...", flush=True)
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.raw")
spark.sql(f"DROP TABLE IF EXISTS {SPARK_TABLE}")
log(f"DROP TABLE IF EXISTS {SPARK_TABLE}")


[SPARK] SparkSession starten ...

[SPARK] Namespace und Cleanup ...
  DROP TABLE IF EXISTS nessie.raw.yellow_taxi_200901_spark


In [26]:
print(f"\n[SPARK] {SOURCE_FILE} -> {SPARK_TABLE}", flush=True)

taxi_df_spark = spark.read.parquet(SOURCE_FILE)

log("Schema (erste 8 Spalten):")
taxi_df_spark.select(taxi_df_spark.columns[:8]).printSchema()

start = time.perf_counter()

(
    taxi_df_spark.writeTo(SPARK_TABLE)
    .tableProperty("write.format.default", "parquet")
    .tableProperty("location", f"{SPARK_RAW_BUCKET}/yellow_taxi_200901_spark")
    .create()
)

elapsed_spark = time.perf_counter() - start

cnt_spark = spark.sql(f"SELECT count(*) AS cnt FROM {SPARK_TABLE}").collect()[0]["cnt"]

print(f"  -> {SPARK_TABLE}: {cnt_spark} Zeilen geladen", flush=True)
print(f"  -> Ingestion-Dauer (Spark): {elapsed_spark:.2f}s ({elapsed_spark/60:.2f}min)", flush=True)

RESULTS["spark"] = {"table": SPARK_TABLE, "rows": cnt_spark, "seconds": elapsed_spark}


[SPARK] ../data/yellow_taxi/yellow_tripdata_2009-01.parquet -> nessie.raw.yellow_taxi_200901_spark
  Schema (erste 8 Spalten):
root
 |-- vendor_name: string (nullable = true)
 |-- Trip_Pickup_DateTime: string (nullable = true)
 |-- Trip_Dropoff_DateTime: string (nullable = true)
 |-- Passenger_Count: long (nullable = true)
 |-- Trip_Distance: double (nullable = true)
 |-- Start_Lon: double (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Rate_Code: double (nullable = true)

  -> nessie.raw.yellow_taxi_200901_spark: 14092413 Zeilen geladen
  -> Ingestion-Dauer (Spark): 161.02s (2.68min)


In [27]:
# Spark-Session schliessen, bevor DuckDB im selben Kernel weitermacht
spark.stop()
log("SparkSession gestoppt")

  SparkSession gestoppt


## Approach 2: DuckDB -> Iceberg via Nessie REST Catalog

**Experimenteller Schreibpfad** (DuckDB 1.5.5 `iceberg`-Extension). Warehouse-Name
`'raw'` ist eine Annahme basierend auf der Bucket-Namenskonvention -- bitte gegen
die Nessie-Server-Config pruefen (`nessie.catalog.warehouses.*`).

In [28]:
!python -m pip install duckdb

In [29]:
import os
import duckdb

MINIO_ENDPOINT_HOST = "minio:9000"
MINIO_ACCESS_KEY = os.environ.get("MINIO_ROOT_USER", "lakehouse")
MINIO_SECRET_KEY = os.environ.get("MINIO_ROOT_PASSWORD", "lakehouse123")

NESSIE_REST_ENDPOINT = "http://nessie:19120/iceberg"
NESSIE_WAREHOUSE = "raw"  # BESTAETIGT: nessie.catalog.warehouses.raw.location=s3://raw/ (docker-compose.yaml)

CATALOG_ALIAS = "nessie"
# Hinweis: das Namespace-Segment "raw" unten ist ein Iceberg-Namespace INNERHALB
# des Warehouse "raw" -- zwei verschiedene Konzepte, die hier zufaellig gleich
# heissen (Warehouse = Storage-Root s3://raw/, Namespace = logischer Ordner darin)
DUCK_TABLE = f"{CATALOG_ALIAS}.raw.yellow_taxi_200901_duck"

print("\n[DUCKDB] Verbindung und Extensions ...", flush=True)
con = duckdb.connect(database=":memory:")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL iceberg; LOAD iceberg;")
log(f"DuckDB Version: {duckdb.__version__}")

con.execute(f"""
    CREATE OR REPLACE SECRET minio_secret (
        TYPE S3,
        KEY_ID '{MINIO_ACCESS_KEY}',
        SECRET '{MINIO_SECRET_KEY}',
        ENDPOINT '{MINIO_ENDPOINT_HOST}',
        URL_STYLE 'path',
        USE_SSL false
    );
""")
log("S3-Secret fuer MinIO angelegt")

con.execute("""
    CREATE OR REPLACE SECRET nessie_iceberg_secret (
        TYPE ICEBERG,
        TOKEN ''
    );
""")
log("Iceberg-Secret angelegt (TOKEN '' -- Nessie ohne Authentifizierung)")

con.execute(f"""
    ATTACH '{NESSIE_WAREHOUSE}' AS {CATALOG_ALIAS} (
        TYPE ICEBERG,
        ENDPOINT '{NESSIE_REST_ENDPOINT}',
        SECRET nessie_iceberg_secret,
        ACCESS_DELEGATION_MODE 'none'
    );
""")

log(f"Nessie REST Catalog attached als '{CATALOG_ALIAS}' (warehouse='{NESSIE_WAREHOUSE}' -> s3://raw/)")

print("\n[DUCKDB] Namespace und Cleanup ...", flush=True)
con.execute(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_ALIAS}.raw;")
con.execute(f"DROP TABLE IF EXISTS {DUCK_TABLE};")
log(f"DROP TABLE IF EXISTS {DUCK_TABLE}")


[DUCKDB] Verbindung und Extensions ...
  DuckDB Version: 1.5.5
  S3-Secret fuer MinIO angelegt
  Iceberg-Secret angelegt (TOKEN '' -- Nessie ohne Authentifizierung)
  Nessie REST Catalog attached als 'nessie' (warehouse='raw' -> s3://raw/)

[DUCKDB] Namespace und Cleanup ...
  DROP TABLE IF EXISTS nessie.raw.yellow_taxi_200901_duck


In [30]:
print(f"\n[DUCKDB] {SOURCE_FILE} -> {DUCK_TABLE}", flush=True)
log("Schema-Vorschau (erste 8 Spalten):")
preview = con.execute(f"SELECT * FROM read_parquet('{SOURCE_FILE}') LIMIT 0").description
for col in preview[:8]:
    print(f"    {col[0]}: {col[1]}", flush=True)

start = time.perf_counter()

con.execute(f"""
    CREATE TABLE {DUCK_TABLE} AS
    SELECT * FROM read_parquet('{SOURCE_FILE}')
""")

elapsed_duck = time.perf_counter() - start

cnt_duck = con.execute(f"SELECT count(*) AS cnt FROM {DUCK_TABLE}").fetchone()[0]

print(f"  -> {DUCK_TABLE}: {cnt_duck} Zeilen geladen", flush=True)
print(f"  -> Ingestion-Dauer (DuckDB): {elapsed_duck:.2f}s ({elapsed_duck/60:.2f}min)", flush=True)

RESULTS["duckdb"] = {"table": DUCK_TABLE, "rows": cnt_duck, "seconds": elapsed_duck}

con.close()


[DUCKDB] ../data/yellow_taxi/yellow_tripdata_2009-01.parquet -> nessie.raw.yellow_taxi_200901_duck
  Schema-Vorschau (erste 8 Spalten):
    vendor_name: VARCHAR
    Trip_Pickup_DateTime: VARCHAR
    Trip_Dropoff_DateTime: VARCHAR
    Passenger_Count: BIGINT
    Trip_Distance: DOUBLE
    Start_Lon: DOUBLE
    Start_Lat: DOUBLE
    Rate_Code: DOUBLE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> nessie.raw.yellow_taxi_200901_duck: 14092413 Zeilen geladen
  -> Ingestion-Dauer (DuckDB): 76.00s (1.27min)


## Vergleich

In [31]:
print("\n" + "=" * 70, flush=True)
print("  Ingestion-Vergleich: Spark vs. DuckDB", flush=True)
print("=" * 70, flush=True)
print(f"  {'Approach':<10} {'Tabelle':<40} {'Zeilen':>10} {'Dauer (s)':>12}", flush=True)
print(f"  {'-'*72}", flush=True)
for approach, r in RESULTS.items():
    print(f"  {approach:<10} {r['table']:<40} {r['rows']:>10} {r['seconds']:>12.2f}", flush=True)

if "spark" in RESULTS and "duckdb" in RESULTS:
    faster = "duckdb" if RESULTS["duckdb"]["seconds"] < RESULTS["spark"]["seconds"] else "spark"
    ratio = max(RESULTS["spark"]["seconds"], RESULTS["duckdb"]["seconds"]) / min(RESULTS["spark"]["seconds"], RESULTS["duckdb"]["seconds"])
    print(f"\n  {faster} war {ratio:.1f}x schneller fuer diese Datei.", flush=True)


  Ingestion-Vergleich: Spark vs. DuckDB
  Approach   Tabelle                                      Zeilen    Dauer (s)
  ------------------------------------------------------------------------
  spark      nessie.raw.yellow_taxi_200901_spark        14092413       161.02
  duckdb     nessie.raw.yellow_taxi_200901_duck         14092413        76.00

  duckdb war 2.1x schneller fuer diese Datei.
